# Module 16 — Week 5 — Bayesian Black-Box Optimisation Capstone

**Strategy: Exploit winners, fix losers, reset failures**

| Function | W4 Result | All-time Best | Status |
|----------|-----------|--------------|--------|
| F1 | -2.66e-133 | 2.82e-04 (W2) | Lock onto W2 point |
| F2 | 0.147 | 0.611 (initial) | Trust on initial best |
| F3 | -0.022 | -0.011 (W1) | Tighter trust on W1 |
| F4 | -0.128 | -0.128 (W4) | Exploit W4 improvement |
| F5 | 2496.35 | 2496.35 (W4) | Tightest exploit |
| F6 | -0.386 | -0.361 (W1) | Full reset — wide LHS |
| F7 | 2.671 | 2.671 (W4) | Tighten trust on W4 |
| F8 | 9.897 | 9.897 (W4) | Slightly wider trust |

In [1]:
import matplotlib
matplotlib.use('Agg')   # headless backend

import numpy as np
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from sklearn.svm import SVC
from scipy.stats import norm
from scipy.stats.qmc import LatinHypercube
import warnings
import os
warnings.filterwarnings('ignore')

PLOTS_DIR = '/Users/luckydhanvi/Documents/DataScience/PracticalLive/LEVEL_2/imperial-aiml-capstone/module-16/plots'
os.makedirs(PLOTS_DIR, exist_ok=True)

print('Module 16 — Week 5 — Bayesian Black-Box Optimisation')
print('All imports loaded successfully')
print(f'Plots will be saved to: {PLOTS_DIR}')

Module 16 — Week 5 — Bayesian Black-Box Optimisation
All imports loaded successfully
Plots will be saved to: /Users/luckydhanvi/Documents/DataScience/PracticalLive/LEVEL_2/imperial-aiml-capstone/module-16/plots


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# ALL HISTORICAL SUBMISSIONS — W1 through W4
# ─────────────────────────────────────────────────────────────────────────────

submitted_x_w1 = {
    1: [0.020584, 0.969910],
    2: [0.814691, 0.969505],
    3: [0.376075, 0.370839, 0.474761],
    4: [0.369789, 0.452786, 0.367951, 0.448446],
    5: [0.241041, 0.805036, 0.948951, 0.905090],
    6: [0.466959, 0.356875, 0.489683, 0.726384, 0.125125],
    7: [0.027698, 0.531762, 0.337094, 0.176133, 0.361503, 0.730849],
    8: [0.192432, 0.183093, 0.018724, 0.036362, 0.690267, 0.444236, 0.081374, 0.428967],
}
new_y_w1 = {
    1:  1.966e-321,
    2:  0.1292261555216582,
    3: -0.010707313301147062,
    4: -0.34595283782499875,
    5:  1450.9433021815964,
    6: -0.3611823990070205,
    7:  1.4058168801082682,
    8:  9.8915570907296,
}

submitted_x_w2 = {
    1: [0.591837, 0.591837],
    2: [0.000000, 1.000000],
    3: [0.421053, 1.000000, 1.000000],
    4: [0.909548, 0.568955, 0.762175, 0.811807],
    5: [0.204881, 0.877830, 0.879582, 0.870578],
    6: [0.851439, 0.906254, 0.506372, 0.594105, 0.708147],
    7: [0.097054, 0.432660, 0.338116, 0.122619, 0.296117, 0.886436],
    8: [0.076274, 0.101214, 0.383035, 0.338493, 0.113685, 0.882235, 0.615428, 0.796463],
}
new_y_w2 = {
    1:  0.00028209052469858225,
    2:  0.1709619176069506,
    3: -0.48304244384724265,
    4: -26.59459580774249,
    5:  1192.2995655092311,
    6: -1.9259411859252866,
    7:  1.2030170341293975,
    8:  9.0382459830856,
}

submitted_x_w3 = {
    1: [0.980000, 0.980000],
    2: [1.000000, 0.306122],
    3: [1.000000, 0.000000, 0.684211],
    4: [0.985601, 0.686679, 0.243615, 0.798556],
    5: [0.204881, 0.877830, 0.879582, 0.870578],  # duplicate of W2
    6: [0.061416, 0.762464, 0.106527, 0.271402, 0.782742],
    7: [0.067189, 0.412831, 0.295130, 0.070570, 0.412599, 0.616173],
    8: [0.682757, 0.427203, 0.591529, 0.734064, 0.514947, 0.813984, 0.722156, 0.615073],
}
new_y_w3 = {
    1:  2.665897212344236e-174,
    2: -0.042550557700427774,
    3: -0.1840890683677661,
    4: -26.07041694623693,
    5:  1192.2995655092311,
    6: -2.508952125110497,
    7:  1.2533263563752521,
    8:  7.5792591902086,
}

# Week 4 submissions and results
submitted_x_w4 = {
    1: [0.278296, 0.020000],
    2: [0.685269, 0.947006],
    3: [0.403468, 0.441923, 0.497061],
    4: [0.352971, 0.651614, 0.805417, 0.616108],
    5: [0.167299, 0.881015, 0.978872, 0.954244],
    6: [0.334649, 0.293944, 0.500782, 0.769829, 0.074923],
    7: [0.206363, 0.281987, 0.389442, 0.281544, 0.218827, 0.711599],
    8: [0.095545, 0.327238, 0.051339, 0.269531, 0.555763, 0.417489, 0.285113, 0.613881],
}
new_y_w4 = {
    1: -2.6647756688938686e-133,
    2:  0.14705786268424045,
    3: -0.022992940111015336,
    4: -0.1283640964538999,
    5:  2496.347187728138,
    6: -0.38592078647528016,
    7:  2.6705394912160187,
    8:  9.8967959631939,
}

print('All historical data loaded — W1 / W2 / W3 / W4')
print()
print('W4 results summary:')
all_time_best_before_w4 = {
    1: 0.00028209052469858225,   # W2
    2: 0.611205,                  # initial data
    3: -0.010707313301147062,     # W1
    4: -0.34595283782499875,      # W1
    5: 1450.9433021815964,        # W1
    6: -0.3611823990070205,       # W1
    7: 1.4058168801082682,        # W1
    8: 9.8915570907296,           # W1
}
for i in range(1,9):
    w4 = new_y_w4[i]
    prev = all_time_best_before_w4[i]
    verdict = 'IMPROVED' if w4 > prev else 'regressed'
    print(f'  F{i}: W4={w4:.4e}  prev_best={prev:.4e}  {verdict}')

All historical data loaded — W1 / W2 / W3 / W4

W4 results summary:
  F1: W4=-2.6648e-133  prev_best=2.8209e-04  regressed
  F2: W4=1.4706e-01  prev_best=6.1120e-01  regressed
  F3: W4=-2.2993e-02  prev_best=-1.0707e-02  regressed
  F4: W4=-1.2836e-01  prev_best=-3.4595e-01  IMPROVED
  F5: W4=2.4963e+03  prev_best=1.4509e+03  IMPROVED
  F6: W4=-3.8592e-01  prev_best=-3.6118e-01  regressed
  F7: W4=2.6705e+00  prev_best=1.4058e+00  IMPROVED
  F8: W4=9.8968e+00  prev_best=9.8916e+00  IMPROVED


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Load initial .npy data and stack all weekly observations (14 pts per fn)
# ─────────────────────────────────────────────────────────────────────────────
base_path = '/Users/luckydhanvi/Documents/DataScience/PracticalLive/LEVEL_2/imperial-aiml-capstone/data/'

descriptions = {
    1: 'Radiation Detection',
    2: 'Noisy ML Model',
    3: 'Drug Discovery',
    4: 'Warehouse Placement',
    5: 'Chemical Yield (STAR)',
    6: 'Cake Recipe',
    7: 'ML Hyperparameters',
    8: 'Complex 8D',
}

data = {}
for i in range(1, 9):
    X0 = np.load(f'{base_path}function_{i}/initial_inputs.npy')
    Y0 = np.load(f'{base_path}function_{i}/initial_outputs.npy')

    X_all = np.vstack([
        X0,
        np.array(submitted_x_w1[i]).reshape(1, -1),
        np.array(submitted_x_w2[i]).reshape(1, -1),
        np.array(submitted_x_w3[i]).reshape(1, -1),
        np.array(submitted_x_w4[i]).reshape(1, -1),
    ])
    Y_all = np.concatenate([Y0, [new_y_w1[i]], [new_y_w2[i]], [new_y_w3[i]], [new_y_w4[i]]])

    data[i] = {'X': X_all, 'Y': Y_all}

# All-time best per function (updated with W4)
all_time_best = {}
for i in range(1, 9):
    Y = data[i]['Y']
    best_idx = np.argmax(Y)
    all_time_best[i] = {'Y': Y[best_idx], 'X': data[i]['X'][best_idx]}

print(f'{"Fn":<4} {"Description":<24} {"N":<5} {"Dim":<5} {"All-time Best Y"}')
print('-' * 65)
for i in range(1, 9):
    Y   = data[i]['Y']
    dim = data[i]['X'].shape[1]
    print(f'F{i:<3} {descriptions[i]:<24} {len(Y):<5} {dim:<5} {all_time_best[i]["Y"]:.6e}')
print()
print('14 observations per function (10 initial + W1 + W2 + W3 + W4)')

Fn   Description              N     Dim   All-time Best Y
-----------------------------------------------------------------
F1   Radiation Detection      14    2     2.820905e-04
F2   Noisy ML Model           14    2     6.112052e-01
F3   Drug Discovery           19    3     -1.070731e-02
F4   Warehouse Placement      34    4     -1.283641e-01
F5   Chemical Yield (STAR)    24    4     2.496347e+03
F6   Cake Recipe              24    5     -3.611824e-01
F7   ML Hyperparameters       34    6     2.670539e+00
F8   Complex 8D               44    8     9.896796e+00

14 observations per function (10 initial + W1 + W2 + W3 + W4)


In [4]:
BOUND_LO, BOUND_HI = 0.02, 0.98

# Track ALL submitted points including W4
all_submitted_X = {
    i: [
        np.array(submitted_x_w1[i]),
        np.array(submitted_x_w2[i]),
        np.array(submitted_x_w3[i]),
        np.array(submitted_x_w4[i]),
    ]
    for i in range(1, 9)
}

def check_duplicate(next_x, func_num, threshold=0.015):
    """True if next_x is within threshold of any prior submission."""
    return any(np.linalg.norm(next_x - x) < threshold for x in all_submitted_X[func_num])

def expected_improvement(mu, sigma, best_y_log, xi=0.01):
    imp = mu - best_y_log - xi
    Z   = imp / (sigma + 1e-9)
    ei  = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    ei[sigma < 1e-10] = 0.0
    return ei

def gp_predict_scalar(gp, x):
    return float(gp.predict(x.reshape(1, -1)).ravel()[0])

def build_search_grid(dim, trust_center=None, trust_radius=None, use_lhs=False, n=50000):
    N = n
    if trust_center is not None and trust_radius is not None:
        lo = np.clip(trust_center - trust_radius, BOUND_LO, BOUND_HI)
        hi = np.clip(trust_center + trust_radius, BOUND_LO, BOUND_HI)
        s  = LatinHypercube(d=dim, seed=42).random(n=N)
        return lo + s*(hi-lo), f'Trust-LHS r={trust_radius:.2f} {dim}D {N:,}pts'
    elif dim == 2:
        g = np.linspace(BOUND_LO, BOUND_HI, 224)
        XX, YY = np.meshgrid(g, g)
        return np.column_stack([XX.ravel(), YY.ravel()]), '224x224 grid ~50k pts'
    elif use_lhs:
        s = LatinHypercube(d=dim, seed=42).random(n=N)
        return BOUND_LO + s*(BOUND_HI-BOUND_LO), f'LHS {N:,}pts {dim}D'
    else:
        np.random.seed(42)
        return np.random.uniform(BOUND_LO, BOUND_HI, (N, dim)), f'Random {N:,}pts {dim}D'

def analyse_function_w5(func_num, beta_ucb=2.0, use_ei=True,
                         use_lhs=False, trust_center=None, trust_radius=None,
                         xi=0.01, length_scale=0.2,
                         remove_outliers=False, outlier_threshold=None):
    X, Y = data[func_num]['X'], data[func_num]['Y']
    dim      = X.shape[1]
    best_idx = np.argmax(Y)
    best_X   = X[best_idx] if trust_center is None else trust_center
    best_Y   = Y[best_idx]

    print(f'\n{"="*68}')
    print(f'F{func_num} {descriptions[func_num]} | Dim={dim} N={len(Y)} BestY={best_Y:.4e}')
    print(f'Best X* = [{" ".join(f"{v:.4f}" for v in X[np.argmax(Y)])}]')
    print(f'W4 result = {new_y_w4[func_num]:.4e}')
    print('='*68)

    # Outlier removal
    X_fit, Y_fit_raw = X, Y
    if remove_outliers and outlier_threshold is not None:
        mask = Y > outlier_threshold
        X_fit, Y_fit_raw = X[mask], Y[mask]
        print(f'  [Outlier] Removed {(~mask).sum()} pts (Y<{outlier_threshold}) — GP on {mask.sum()} pts')

    # Build search grid
    X_grid, grid_info = build_search_grid(
        dim,
        trust_center=best_X if trust_radius else None,
        trust_radius=trust_radius,
        use_lhs=use_lhs
    )
    print(f'  [Grid] {grid_info}')

    # Fit GP on log-transformed outputs
    Y_log  = np.log(np.abs(Y_fit_raw) + 1e-300) * np.sign(Y_fit_raw + 1e-300)
    kernel = Matern(length_scale=length_scale, nu=2.5)
    gp     = GaussianProcessRegressor(kernel=kernel, alpha=1e-6,
                                       n_restarts_optimizer=3, normalize_y=True)
    gp.fit(X_fit, Y_log)

    # Sanity check
    mu_chk, std_chk = gp.predict(X[np.argmax(Y)].reshape(1,-1), return_std=True)
    actual_log = np.log(np.abs(best_Y)+1e-300)*np.sign(best_Y+1e-300)
    print(f'  [GP] {gp.kernel_}')
    print(f'  [Sanity] pred={float(mu_chk.ravel()[0]):.4f} actual={actual_log:.4f} std={float(std_chk.ravel()[0]):.6f}')

    # UCB + EI ensemble
    mu, sigma = gp.predict(X_grid, return_std=True)
    best_y_log = np.log(np.abs(best_Y)+1e-300)*np.sign(best_Y+1e-300)

    ucb   = mu + beta_ucb * sigma
    x_ucb = X_grid[np.argmax(ucb)]

    ei    = expected_improvement(mu, sigma, best_y_log, xi=xi)
    x_ei  = X_grid[np.argmax(ei)]

    print(f'  [UCB] b={beta_ucb} max={ucb.max():.4f} => [{" ".join(f"{v:.4f}" for v in x_ucb)}]')
    print(f'  [EI]  xi={xi} max={ei.max():.6f} => [{" ".join(f"{v:.4f}" for v in x_ei)}]')

    mu_u = gp_predict_scalar(gp, x_ucb)
    mu_e = gp_predict_scalar(gp, x_ei)
    next_x, winner = (x_ei, 'EI') if (use_ei and mu_e >= mu_u) else (x_ucb, 'UCB')
    print(f'  [Ensemble] UCB_mean={mu_u:.4f} EI_mean={mu_e:.4f} => winner={winner}')

    # Duplicate check
    if check_duplicate(next_x, func_num):
        np.random.seed(99)
        next_x = np.clip(next_x + np.random.uniform(-0.03, 0.03, dim), BOUND_LO, BOUND_HI)
        print(f'  [Dup] Duplicate detected — point perturbed')

    print(f'  [Dist] Distance from best: {np.linalg.norm(next_x - X[np.argmax(Y)]):.4f}')
    portal = '-'.join([f'{v:.6f}' for v in next_x])
    print(f'\n  >>> SUBMIT F{func_num}: {portal} <<<')
    return next_x, portal

print('Helpers ready — W5 version (tracks W1-W4 submissions)')

Helpers ready — W5 version (tracks W1-W4 submissions)


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# F1 — Radiation Detection (2D)
# W4 result: -2.66e-133 — complete disaster, wide exploration failed again
# All-time best: 2.82e-04 (W2) at [0.591837, 0.591837]
# Strategy W5: LOCK onto W2 point — tight trust region r=0.10, low beta
#              Stop all exploration. Only exploit the one point that worked.
# ─────────────────────────────────────────────────────────────────────────────
W2_BEST_X1 = np.array([0.591837, 0.591837])  # gave 2.82e-04

next_x1, portal1 = analyse_function_w5(
    func_num     = 1,
    beta_ucb     = 1.0,          # low — exploit, don't explore
    use_ei       = True,
    use_lhs      = False,        # 2D trust grid
    trust_center = W2_BEST_X1,   # anchor to W2 best, NOT current GP best
    trust_radius = 0.10,         # tight neighbourhood
    xi           = 0.001,
)
print(f'  [W5-F1] Anchored to W2 point {W2_BEST_X1} (gave 2.82e-04)')


F1 Radiation Detection | Dim=2 N=14 BestY=2.8209e-04
Best X* = [0.5918 0.5918]
W4 result = -2.6648e-133
  [Grid] Trust-LHS r=0.10 2D 50,000pts
  [GP] Matern(length_scale=0.385, nu=2.5)
  [Sanity] pred=-8.1726 actual=-8.1733 std=0.242036
  [UCB] b=1.0 max=31.5087 => [0.5445 0.6820]
  [EI]  xi=0.001 max=17.418199 => [0.5801 0.6832]
  [Ensemble] UCB_mean=-2.6202 EI_mean=5.0208 => winner=EI
  [Dist] Distance from best: 0.0921

  >>> SUBMIT F1: 0.580092-0.683225 <<<
  [W5-F1] Anchored to W2 point [0.591837 0.591837] (gave 2.82e-04)


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# F2 — Noisy ML Model (2D)
# W4 result: 0.147 — still below W2 (0.171) and initial best (0.611)
# All-time best: 0.611 (initial data) at [0.70263656, 0.9265642]
# Strategy W5: Trust region anchored to the initial data point that gave 0.611
#              We've been drifting — go back to where it actually worked
# ─────────────────────────────────────────────────────────────────────────────
INIT_BEST_X2 = np.array([0.70263656, 0.9265642])  # gave 0.611 in initial data

next_x2, portal2 = analyse_function_w5(
    func_num     = 2,
    beta_ucb     = 0.5,           # very exploitative
    use_ei       = True,
    use_lhs      = False,
    trust_center = INIT_BEST_X2,  # anchor to initial best
    trust_radius = 0.15,
    xi           = 0.01,
)
print(f'  [W5-F2] Anchored to initial best {INIT_BEST_X2} (gave 0.611)')


F2 Noisy ML Model | Dim=2 N=14 BestY=6.1121e-01
Best X* = [0.7026 0.9266]
W4 result = 1.4706e-01
  [Grid] Trust-LHS r=0.15 2D 50,000pts
  [GP] Matern(length_scale=1e-05, nu=2.5)
  [Sanity] pred=-0.4923 actual=-0.4923 std=0.002287
  [UCB] b=0.5 max=0.5195 => [0.7028 0.9266]
  [EI]  xi=0.01 max=0.843262 => [0.7495 0.8532]
  [Ensemble] UCB_mean=-0.6239 EI_mean=-0.6239 => winner=UCB
  [Dist] Distance from best: 0.0002

  >>> SUBMIT F2: 0.702813-0.926626 <<<
  [W5-F2] Anchored to initial best [0.70263656 0.9265642 ] (gave 0.611)


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# F3 — Drug Discovery (3D)
# W4 result: -0.022 — regression from W1 best (-0.011)
# All-time best: -0.011 (W1) at [0.376075, 0.370839, 0.474761]
# Strategy W5: Tighter trust on W1 point — r=0.12 (was 0.20), lower beta
# ─────────────────────────────────────────────────────────────────────────────
W1_BEST_X3 = np.array([0.376075, 0.370839, 0.474761])  # gave -0.011

next_x3, portal3 = analyse_function_w5(
    func_num     = 3,
    beta_ucb     = 1.0,          # reduced from 1.5
    use_ei       = True,
    use_lhs      = True,         # LHS within trust region for 3D
    trust_center = W1_BEST_X3,   # anchor to W1 best
    trust_radius = 0.12,         # tighter than W4 (was 0.20)
    xi           = 0.001,
)
print(f'  [W5-F3] Anchored to W1 best {W1_BEST_X3} (gave -0.011)')


F3 Drug Discovery | Dim=3 N=19 BestY=-1.0707e-02
Best X* = [0.3761 0.3708 0.4748]
W4 result = -2.2993e-02
  [Grid] Trust-LHS r=0.12 3D 50,000pts
  [GP] Matern(length_scale=0.157, nu=2.5)
  [Sanity] pred=4.5368 actual=4.5368 std=0.000889
  [UCB] b=1.0 max=4.8420 => [0.3659 0.3062 0.4790]
  [EI]  xi=0.001 max=0.118576 => [0.3651 0.3164 0.4710]
  [Ensemble] UCB_mean=4.4806 EI_mean=4.5359 => winner=EI
  [Dist] Distance from best: 0.0556

  >>> SUBMIT F3: 0.365086-0.316421-0.471038 <<<
  [W5-F3] Anchored to W1 best [0.376075 0.370839 0.474761] (gave -0.011)


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# F4 — Warehouse Placement (4D)
# W4 result: -0.128 — FIRST EVER IMPROVEMENT (was -0.346)
# All-time best: -0.128 (W4) at [0.352971, 0.651614, 0.805417, 0.616108]
# Strategy W5: Exploit W4 improvement — trust region around W4 best point
#              Keep outlier removal — W2/W3 disasters still corrupt GP
# ─────────────────────────────────────────────────────────────────────────────
next_x4, portal4 = analyse_function_w5(
    func_num          = 4,
    beta_ucb          = 1.5,
    use_ei            = True,
    use_lhs           = True,
    trust_radius      = 0.20,    # exploit W4 neighbourhood
    xi                = 0.01,
    remove_outliers   = True,
    outlier_threshold = -5.0,    # removes W2(-26.59) and W3(-26.07)
)


F4 Warehouse Placement | Dim=4 N=34 BestY=-1.2836e-01
Best X* = [0.3530 0.6516 0.8054 0.6161]
W4 result = -1.2836e-01
  [Outlier] Removed 31 pts (Y<-5.0) — GP on 3 pts
  [Grid] Trust-LHS r=0.20 4D 50,000pts
  [GP] Matern(length_scale=0.00263, nu=2.5)
  [Sanity] pred=2.0529 actual=2.0529 std=0.001448
  [UCB] b=1.5 max=2.7463 => [0.3447 0.6455 0.7920 0.6225]
  [EI]  xi=0.01 max=0.114344 => [0.3447 0.6455 0.7920 0.6225]
  [Ensemble] UCB_mean=0.5739 EI_mean=0.5739 => winner=EI
  [Dist] Distance from best: 0.0181

  >>> SUBMIT F4: 0.344700-0.645505-0.791987-0.622463 <<<


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# F5 — Chemical Yield (4D) — STAR PERFORMER
# W4 result: 2496.35 — massive jump from 1450.94
# All-time best: 2496.35 (W4) at [0.167299, 0.881015, 0.978872, 0.954244]
# Strategy W5: TIGHTEST exploitation — r=0.05 (was 0.08), beta=0.05
#              Do not wander. Squeeze the peak harder.
# ─────────────────────────────────────────────────────────────────────────────
W4_BEST_X5 = np.array([0.167299, 0.881015, 0.978872, 0.954244])  # gave 2496.35

next_x5, portal5 = analyse_function_w5(
    func_num     = 5,
    beta_ucb     = 0.05,         # almost pure exploitation
    use_ei       = True,
    use_lhs      = True,
    trust_center = W4_BEST_X5,   # anchor to W4 best
    trust_radius = 0.05,         # tighter than W4 (was 0.08)
    xi           = 0.01,
)
print(f'  [W5-F5] Anchored to W4 best {W4_BEST_X5} (gave 2496.35)')


F5 Chemical Yield (STAR) | Dim=4 N=24 BestY=2.4963e+03
Best X* = [0.1673 0.8810 0.9789 0.9542]
W4 result = 2.4963e+03
  [Grid] Trust-LHS r=0.05 4D 50,000pts
  [GP] Matern(length_scale=0.395, nu=2.5)
  [Sanity] pred=7.8226 actual=7.8226 std=0.002193


  [UCB] b=0.05 max=7.8749 => [0.1396 0.9115 0.9799 0.9770]
  [EI]  xi=0.01 max=0.161898 => [0.2168 0.9304 0.9687 0.9793]
  [Ensemble] UCB_mean=7.8657 EI_mean=7.7489 => winner=UCB
  [Dist] Distance from best: 0.0471

  >>> SUBMIT F5: 0.139557-0.911522-0.979905-0.977049 <<<
  [W5-F5] Anchored to W4 best [0.167299 0.881015 0.978872 0.954244] (gave 2496.35)


In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# F6 — Cake Recipe (5D)
# W4 result: -0.386 — worse than W1 best (-0.361). 4 weeks of regression.
# All-time best: -0.361 (W1) at [0.466959, 0.356875, 0.489683, 0.726384, 0.125125]
# Strategy W5: FULL RESET — wide LHS across the full space.
#              Current approach is not working. Cast the net wide again.
#              Also try trust region on W1 best — pick whichever GP predicts higher.
# ─────────────────────────────────────────────────────────────────────────────
print('F6 — Trying two strategies and picking the better GP prediction...')
print()
print('--- Strategy A: Wide LHS (full reset) ---')
next_x6a, portal6a = analyse_function_w5(
    func_num     = 6,
    beta_ucb     = 2.0,     # exploration — full reset
    use_ei       = True,
    use_lhs      = True,    # wide LHS no trust region
    trust_radius = None,
    xi           = 0.001,
)

print()
print('--- Strategy B: Trust region on W1 best ---')
W1_BEST_X6 = np.array([0.466959, 0.356875, 0.489683, 0.726384, 0.125125])  # gave -0.361
next_x6b, portal6b = analyse_function_w5(
    func_num     = 6,
    beta_ucb     = 1.5,
    use_ei       = True,
    use_lhs      = True,
    trust_center = W1_BEST_X6,
    trust_radius = 0.18,
    xi           = 0.001,
)

# Load GP on all F6 data and compare predicted means
X6, Y6 = data[6]['X'], data[6]['Y']
Y6_log = np.log(np.abs(Y6)+1e-300)*np.sign(Y6+1e-300)
gp6_eval = GaussianProcessRegressor(
    kernel=Matern(length_scale=0.2, nu=2.5), alpha=1e-6,
    n_restarts_optimizer=3, normalize_y=True
)
gp6_eval.fit(X6, Y6_log)
mu_a = gp_predict_scalar(gp6_eval, next_x6a)
mu_b = gp_predict_scalar(gp6_eval, next_x6b)
print(f'\n  [F6 Comparison] Strategy A GP mean={mu_a:.4f} | Strategy B GP mean={mu_b:.4f}')
if mu_a >= mu_b:
    next_x6, portal6 = next_x6a, portal6a
    print('  => SELECTED: Strategy A (Wide LHS)')
else:
    next_x6, portal6 = next_x6b, portal6b
    print('  => SELECTED: Strategy B (W1 Trust Region)')
print(f'\n  >>> FINAL F6: {portal6} <<<')

F6 — Trying two strategies and picking the better GP prediction...

--- Strategy A: Wide LHS (full reset) ---

F6 Cake Recipe | Dim=5 N=24 BestY=-3.6118e-01
Best X* = [0.4670 0.3569 0.4897 0.7264 0.1251]
W4 result = -3.8592e-01


  [Grid] LHS 50,000pts 5D


  [GP] Matern(length_scale=0.596, nu=2.5)
  [Sanity] pred=1.0184 actual=1.0184 std=0.000493
  [UCB] b=2.0 max=1.2016 => [0.3707 0.3694 0.4987 0.5059 0.0256]
  [EI]  xi=0.001 max=0.018382 => [0.4551 0.3759 0.5780 0.6741 0.0220]
  [Ensemble] UCB_mean=0.7880 EI_mean=0.9478 => winner=EI
  [Dist] Distance from best: 0.1472

  >>> SUBMIT F6: 0.455127-0.375851-0.577994-0.674115-0.022034 <<<

--- Strategy B: Trust region on W1 best ---

F6 Cake Recipe | Dim=5 N=24 BestY=-3.6118e-01
Best X* = [0.4670 0.3569 0.4897 0.7264 0.1251]
W4 result = -3.8592e-01
  [Grid] Trust-LHS r=0.18 5D 50,000pts
  [GP] Matern(length_scale=0.596, nu=2.5)
  [Sanity] pred=1.0184 actual=1.0184 std=0.000493


  [UCB] b=1.5 max=1.1535 => [0.3936 0.3768 0.4712 0.6408 0.0259]
  [EI]  xi=0.001 max=0.027546 => [0.4178 0.3570 0.4681 0.6685 0.0395]
  [Ensemble] UCB_mean=0.9635 EI_mean=0.9951 => winner=EI
  [Dist] Distance from best: 0.1164

  >>> SUBMIT F6: 0.417831-0.356959-0.468069-0.668531-0.039515 <<<

  [F6 Comparison] Strategy A GP mean=0.9478 | Strategy B GP mean=0.9951
  => SELECTED: Strategy B (W1 Trust Region)

  >>> FINAL F6: 0.417831-0.356959-0.468069-0.668531-0.039515 <<<


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# F7 — ML Hyperparameter Tuning (6D)
# W4 result: 2.671 — great improvement from 1.406
# All-time best: 2.671 (W4) at [0.206363, 0.281987, 0.389442, 0.281544, 0.218827, 0.711599]
# Strategy W5: Tighten trust region (r=0.18, was 0.25), lower beta (1.2, was 2.0)
#              Squeeze harder around the W4 winning region
# ─────────────────────────────────────────────────────────────────────────────
next_x7, portal7 = analyse_function_w5(
    func_num     = 7,
    beta_ucb     = 1.2,          # lower than W4 (was 2.0)
    use_ei       = True,
    use_lhs      = True,
    trust_radius = 0.18,         # tighter than W4 (was 0.25)
    xi           = 0.01,
)


F7 ML Hyperparameters | Dim=6 N=34 BestY=2.6705e+00
Best X* = [0.2064 0.2820 0.3894 0.2815 0.2188 0.7116]
W4 result = 2.6705e+00
  [Grid] Trust-LHS r=0.18 6D 50,000pts
  [GP] Matern(length_scale=0.576, nu=2.5)
  [Sanity] pred=0.9823 actual=0.9823 std=0.001946


  [UCB] b=1.2 max=1.8073 => [0.3637 0.1115 0.4487 0.3679 0.0581 0.8802]
  [EI]  xi=0.01 max=0.243418 => [0.3841 0.1221 0.4449 0.3571 0.1474 0.7831]
  [Ensemble] UCB_mean=0.5994 EI_mean=0.8486 => winner=EI
  [Dist] Distance from best: 0.2759

  >>> SUBMIT F7: 0.384097-0.122113-0.444891-0.357064-0.147383-0.783086 <<<


In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# F8 — Complex 8D
# W4 result: 9.897 — tiny improvement over 9.892
# All-time best: 9.897 (W4) at [0.095545, 0.327238, 0.051339, 0.269531, 0.555763, 0.417489, 0.285113, 0.613881]
# Strategy W5: Slightly wider trust radius (r=0.30, was 0.25) to find more gain
#              8D landscape needs more room to find improvements
# ─────────────────────────────────────────────────────────────────────────────
next_x8, portal8 = analyse_function_w5(
    func_num     = 8,
    beta_ucb     = 1.5,
    use_ei       = True,
    use_lhs      = True,
    trust_radius = 0.30,         # slightly wider than W4 (was 0.25)
    xi           = 0.01,
)


F8 Complex 8D | Dim=8 N=44 BestY=9.8968e+00
Best X* = [0.0955 0.3272 0.0513 0.2695 0.5558 0.4175 0.2851 0.6139]
W4 result = 9.8968e+00


  [Grid] Trust-LHS r=0.30 8D 50,000pts
  [GP] Matern(length_scale=1.25, nu=2.5)
  [Sanity] pred=2.2922 actual=2.2922 std=0.000131


  [UCB] b=1.5 max=2.3489 => [0.0338 0.0706 0.1545 0.2986 0.8262 0.7058 0.0754 0.6289]
  [EI]  xi=0.01 max=0.010152 => [0.0898 0.0683 0.1810 0.3273 0.7662 0.6534 0.1748 0.4992]
  [Ensemble] UCB_mean=2.2883 EI_mean=2.2965 => winner=EI
  [Dist] Distance from best: 0.4609

  >>> SUBMIT F8: 0.089787-0.068251-0.180968-0.327284-0.766207-0.653365-0.174832-0.499246 <<<


In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# WEEK 5 SUBMISSION SUMMARY
# ─────────────────────────────────────────────────────────────────────────────
print('='*70)
print('WEEK 5 — MODULE 16 — PORTAL SUBMISSION STRINGS')
print('='*70)
portals = {
    1: portal1, 2: portal2, 3: portal3, 4: portal4,
    5: portal5, 6: portal6, 7: portal7, 8: portal8
}
for i in range(1, 9):
    print(f'F{i}: {portals[i]}')
print()
print('All-time bests going into W5:')
best_labels = {1:'W2', 2:'Initial', 3:'W1', 4:'W4', 5:'W4', 6:'W1', 7:'W4', 8:'W4'}
best_vals   = {1:2.82e-4, 2:0.611, 3:-0.011, 4:-0.128, 5:2496.35, 6:-0.361, 7:2.671, 8:9.897}
for i in range(1, 9):
    print(f'  F{i}: {best_vals[i]:.4e} ({best_labels[i]})')

WEEK 5 — MODULE 16 — PORTAL SUBMISSION STRINGS
F1: 0.580092-0.683225
F2: 0.702813-0.926626
F3: 0.365086-0.316421-0.471038
F4: 0.344700-0.645505-0.791987-0.622463
F5: 0.139557-0.911522-0.979905-0.977049
F6: 0.417831-0.356959-0.468069-0.668531-0.039515
F7: 0.384097-0.122113-0.444891-0.357064-0.147383-0.783086
F8: 0.089787-0.068251-0.180968-0.327284-0.766207-0.653365-0.174832-0.499246

All-time bests going into W5:
  F1: 2.8200e-04 (W2)
  F2: 6.1100e-01 (Initial)
  F3: -1.1000e-02 (W1)
  F4: -1.2800e-01 (W4)
  F5: 2.4963e+03 (W4)
  F6: -3.6100e-01 (W1)
  F7: 2.6710e+00 (W4)
  F8: 9.8970e+00 (W4)


In [14]:
# ─────────────────────────────────────────────────────────────────────────────
# Progress plot — W1 through W4 running best per function
# ─────────────────────────────────────────────────────────────────────────────
weekly_results = {
    1: [new_y_w1[i] for i in range(1,9)],
    2: [new_y_w2[i] for i in range(1,9)],
    3: [new_y_w3[i] for i in range(1,9)],
    4: [new_y_w4[i] for i in range(1,9)],
}

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle('Capstone Progress — Weekly Query Value vs Running Best (W1–W4)', fontsize=14, fontweight='bold')

for idx, fn in enumerate(range(1, 9)):
    ax = axes[idx // 4][idx % 4]
    weeks = [1, 2, 3, 4]
    vals  = [weekly_results[w][fn-1] for w in weeks]
    running_best = [max(vals[:w]) for w in range(1, len(vals)+1)]

    ax.plot(weeks, vals, 'o--', color='steelblue', label='Weekly query', alpha=0.7)
    ax.plot(weeks, running_best, 's-', color='darkorange', linewidth=2, label='Running best')
    ax.set_title(f'F{fn}: {descriptions[fn]}', fontsize=9)
    ax.set_xlabel('Week')
    ax.set_ylabel('Output')
    ax.legend(fontsize=7)
    ax.grid(True, alpha=0.3)
    ax.set_xticks([1, 2, 3, 4])

plt.tight_layout()
plot_path = os.path.join(PLOTS_DIR, 'w4_progress_analysis.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.close()
print(f'Progress plot saved to: {plot_path}')

Progress plot saved to: /Users/luckydhanvi/Documents/DataScience/PracticalLive/LEVEL_2/imperial-aiml-capstone/module-16/plots/w4_progress_analysis.png
